# FASE 5 — Predictive Maintenance
## Machine Failure Prediction: Logistic Regression vs Random Forest

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
df = pd.read_csv('../data/raw_data.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [ ]:
# Create failure labels (top 5% stress)
df['stress'] = df['temperature_c']*0.5 + df['vibration_mm_s']*10
threshold = df['stress'].quantile(0.95)
df['failure'] = (df['stress'] >= threshold).astype(int)
print(f'Threshold: {threshold:.2f}')
print(f'Failure=1: {df["failure"].sum()} ({df["failure"].mean()*100:.1f}%)')
print(f'Failure=0: {(df["failure"]==0).sum()} ({(df["failure"]==0).mean()*100:.1f}%)')

In [ ]:
features = ['temperature_c','vibration_mm_s','current_a','voltage_v','power_kw']
X, y = df[features], df['failure']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
Xs_tr, Xs_te = scaler.fit_transform(X_tr), scaler.transform(X_te)
print(f'Train: {len(X_tr)}, Test: {len(X_te)}')

In [ ]:
models = {'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
          'Random Forest': RandomForestClassifier(n_estimators=150, random_state=42)}
results = {}
for name, m in models.items():
    m.fit(Xs_tr, y_tr)
    yp = m.predict(Xs_te)
    results[name] = {'Accuracy': accuracy_score(y_te,yp)*100,
                     'Precision': precision_score(y_te,yp,zero_division=0)*100,
                     'Recall': recall_score(y_te,yp,zero_division=0)*100,
                     'F1': f1_score(y_te,yp,zero_division=0)*100}
    print(f'\n{name}:')
    print(f'  Accuracy:  {results[name]["Accuracy"]:.1f}%')
    print(f'  Precision: {results[name]["Precision"]:.1f}%')
    print(f'  Recall:    {results[name]["Recall"]:.1f}%')
    print(f'  F1 Score:  {results[name]["F1"]:.1f}%')

In [ ]:
rf = models['Random Forest']
df['failure_prob'] = rf.predict_proba(scaler.transform(df[features]))[:,1]
fail_prob = df.groupby('machine_id')['failure_prob'].mean().mul(100).round(1)
print('\nFailure Probability per Machine:')
for m,v in fail_prob.items():
    print(f'  {m}: {v}%')

In [ ]:
# Feature importance
fi = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8,5))
fi.plot(kind='barh', ax=ax, color='#2196F3')
ax.set_title('Feature Importance — Random Forest', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../data/plots/feature_importance.png', dpi=150)
plt.show()

## Results Summary

| Model | Accuracy | Precision | Recall | F1 Score |
|-------|----------|-----------|--------|----------|
| Logistic Regression | 100.0% | 99.5% | 100.0% | 99.8% |
| **Random Forest** | **100.0%** | **99.5%** | **99.5%** | **99.5%** |